# Prewise Qwen3.5-4B — Vietnamese Context Adapters v4

Huấn luyện hai LoRA độc lập: `message-context-adapter` và `explanation-adapter`. Chọn accelerator **GPU T4 x2**.

In [ ]:
from pathlib import Path
import json, subprocess, sys

candidates = [p for p in Path('/kaggle/input').rglob('dataset_manifest.json') if 'v4' in p.read_text(encoding='utf-8')]
assert candidates, 'Không tìm thấy bundle v4 có dataset_manifest.json trong /kaggle/input'
BUNDLE = candidates[0].parent
DATA = Path('/kaggle/working/prewise-data-v4')
WORK = Path('/kaggle/working/prewise-v4')
BASE_MODEL = 'Qwen/Qwen3.5-4B'
manifest = json.loads((BUNDLE / 'dataset_manifest.json').read_text(encoding='utf-8'))
assert manifest['schema_version'] == '4', manifest['schema_version']
print('BUNDLE =', BUNDLE)
print(json.dumps(manifest['coverage'], ensure_ascii=False, indent=2))

## 1. Validate toàn bộ bundle v4

In [ ]:
subprocess.run([sys.executable, str(BUNDLE / 'validate_vi_context_bundle_v4.py'), '--bundle-root', str(BUNDLE)], check=True)

## 2. Giải nén dữ liệu

In [ ]:
subprocess.run([sys.executable, str(BUNDLE / 'prepare_kaggle_data.py'), '--bundle-root', str(BUNDLE), '--output-root', str(DATA)], check=True)

## 3. Cài dependency

In [ ]:
subprocess.run([sys.executable, str(BUNDLE / 'install_kaggle_deps.py')], check=True)

## 4. Train hai adapter và đóng gói

In [ ]:
subprocess.run([
    sys.executable, str(BUNDLE / 'run_kaggle_training_v4.py'),
    '--bundle-root', str(BUNDLE),
    '--data-root', str(DATA),
    '--output-root', str(WORK),
    '--model-name', BASE_MODEL,
    '--message-train-samples', '32799',
    '--explanation-train-samples', '24000',
    '--validation-samples', '2400',
], check=True)

## 5. Kiểm tra file đầu ra

In [ ]:
archive = Path('/kaggle/working/prewise-adapters-v4.zip')
assert archive.exists(), archive
print('READY:', archive, archive.stat().st_size, 'bytes')